In [1]:
!pip install numpy pandas matplotlib scikit-learn xgboost torch joblib

In [3]:
import os
import zipfile
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ZIP = r"E:\MY projects\Flash Flood Prediction Project\data\historical_landslide_monthly.zip"

EXTRACT_DIR = r"E:\MY projects\Flash Flood Prediction Project\data\historical_landslide_monthly"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(DATA_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ ZIP extracted successfully!")

✅ ZIP extracted successfully!


In [4]:
csv_files = glob.glob(
    os.path.join(EXTRACT_DIR, "**", "*.csv"),
    recursive=True
)

print("Number of CSV files:", len(csv_files))

data_list = []

for file in csv_files:

    try:
        if os.path.getsize(file) == 0:
            continue

        temp = pd.read_csv(file)

        if temp.empty:
            continue

        data_list.append(temp)

    except Exception as e:
        print("⚠️ Skipped:", file)
        print("Error:", e)

if len(data_list) == 0:
    raise ValueError("❌ No valid CSV files found!")

df = pd.concat(data_list, ignore_index=True)

print("✅ Dataset loaded")
print("Shape:", df.shape)
print(df.head())

Number of CSV files: 10
✅ Dataset loaded
Shape: (12, 13)
            event_date   latitude  longitude  landslide_type  trigger  source  \
0           2007-07-30  30.557002  79.559999             NaN      NaN     NaN   
1           2015-04-28  30.563098  79.576502             NaN      NaN     NaN   
2           2015-04-30  30.556802  79.554100             NaN      NaN     NaN   
3  2016-09-11 12:55:00  30.596608  79.569769             NaN      NaN     NaN   
4  2017-04-06 13:19:00  30.504157  79.487512             NaN      NaN     NaN   

   location  country  admin_division  fatalities  injuries  year  month  
0       NaN      NaN             NaN         NaN       NaN  2007      7  
1       NaN      NaN             NaN         NaN       NaN  2015      4  
2       NaN      NaN             NaN         NaN       NaN  2015      4  
3       NaN      NaN             NaN         NaN       NaN  2016      9  
4       NaN      NaN             NaN         NaN       NaN  2017      4  


In [5]:
df["event_date"] = pd.to_datetime(
    df["event_date"],
    errors="coerce"
)

df["year"] = pd.to_numeric(
    df["year"],
    errors="coerce"
)

df["month"] = pd.to_numeric(
    df["month"],
    errors="coerce"
)

df = df.dropna(
    subset=["event_date"]
)

df = df.sort_values(
    "event_date"
).reset_index(drop=True)

print("✅ Cleaning completed")
print(df[["event_date", "year", "month"]])

✅ Cleaning completed
  event_date  year  month
0 2007-07-30  2007      7
1 2007-07-30  2007      7
2 2015-04-28  2015      4
3 2015-04-28  2015      4
4 2015-04-30  2015      4
5 2015-04-30  2015      4


In [6]:
monthly_events = (
    df.set_index("event_date")
      .resample("MS")
      .size()
      .to_frame("landslide_count")
)

# Missing months = zero landslides
monthly_events["landslide_count"] = (
    monthly_events["landslide_count"]
    .fillna(0)
)

# Binary occurrence
monthly_events["landslide_occurrence"] = (
    monthly_events["landslide_count"] > 0
).astype(int)

print(monthly_events.head())
print()
print("Total months:", len(monthly_events))
print(
    "Positive months:",
    monthly_events["landslide_occurrence"].sum()
)

            landslide_count  landslide_occurrence
event_date                                       
2007-07-01                2                     1
2007-08-01                0                     0
2007-09-01                0                     0
2007-10-01                0                     0
2007-11-01                0                     0

Total months: 94
Positive months: 2


In [7]:
monthly_events["month"] = monthly_events.index.month
monthly_events["year"] = monthly_events.index.year

print(monthly_events.tail())

            landslide_count  landslide_occurrence  month  year
event_date                                                    
2014-12-01                0                     0     12  2014
2015-01-01                0                     0      1  2015
2015-02-01                0                     0      2  2015
2015-03-01                0                     0      3  2015
2015-04-01                4                     1      4  2015


In [8]:
xgb_df = monthly_events.copy()

# Previous occurrence/count
for lag in [1, 2, 3, 6, 12]:

    xgb_df[f"occurrence_lag_{lag}"] = (
        xgb_df["landslide_occurrence"].shift(lag)
    )

    xgb_df[f"count_lag_{lag}"] = (
        xgb_df["landslide_count"].shift(lag)
    )

# Rolling statistics
xgb_df["rolling_3"] = (
    xgb_df["landslide_occurrence"]
    .rolling(3)
    .sum()
)

xgb_df["rolling_6"] = (
    xgb_df["landslide_occurrence"]
    .rolling(6)
    .sum()
)

xgb_df["rolling_12"] = (
    xgb_df["landslide_occurrence"]
    .rolling(12)
    .sum()
)

# Target = next month's occurrence
xgb_df["target"] = (
    xgb_df["landslide_occurrence"]
    .shift(-1)
)

xgb_df = xgb_df.dropna()

xgb_features = [
    "occurrence_lag_1",
    "occurrence_lag_2",
    "occurrence_lag_3",
    "occurrence_lag_6",
    "occurrence_lag_12",
    "count_lag_1",
    "count_lag_2",
    "count_lag_3",
    "count_lag_6",
    "count_lag_12",
    "rolling_3",
    "rolling_6",
    "rolling_12",
    "month",
    "year"
]

X = xgb_df[xgb_features]
y = xgb_df["target"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive targets:", y.sum())

X shape: (81, 15)
y shape: (81,)
Positive targets: 1


In [9]:
train_size = int(len(X) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())

Training samples: 64
Testing samples: 17
Training positives: 0
Testing positives: 1


In [10]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("✅ XGBoost training completed!")

✅ XGBoost training completed!


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_pred = (
    xgb_prob >= 0.5
).astype(int)

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

xgb_precision = precision_score(
    y_test,
    xgb_pred,
    zero_division=0
)

xgb_recall = recall_score(
    y_test,
    xgb_pred,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_pred,
    zero_division=0
)

if y_test.nunique() == 2:
    xgb_auc = roc_auc_score(
        y_test,
        xgb_prob
    )
else:
    xgb_auc = np.nan

print("XGBoost Results")
print("================")
print("Accuracy :", xgb_accuracy)
print("Precision:", xgb_precision)
print("Recall   :", xgb_recall)
print("F1 Score :", xgb_f1)
print("ROC-AUC  :", xgb_auc)

XGBoost Results
Accuracy : 0.9411764705882353
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 0.5


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

sequence_length = 12

values = monthly_events[
    "landslide_occurrence"
].values.astype(np.float32)

X_seq = []
y_seq = []

for i in range(
    sequence_length,
    len(values)
):

    X_seq.append(
        values[i-sequence_length:i]
    )

    y_seq.append(
        values[i]
    )

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)

X_seq = X_seq.reshape(
    X_seq.shape[0],
    X_seq.shape[1],
    1
)

print("Sequence shape:", X_seq.shape)

Device: cuda
Sequence shape: (82, 12, 1)


In [13]:
train_size_seq = int(
    len(X_seq) * 0.8
)

X_train_seq = X_seq[:train_size_seq]
X_test_seq = X_seq[train_size_seq:]

y_train_seq = y_seq[:train_size_seq]
y_test_seq = y_seq[train_size_seq:]

X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_seq,
    dtype=torch.float32
).unsqueeze(1)

y_test_tensor = torch.tensor(
    y_test_seq,
    dtype=torch.float32
).unsqueeze(1)

In [14]:
class LSTMModel(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=32,
        num_layers=2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out


lstm_model = LSTMModel().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

print("✅ LSTM created")

✅ LSTM created


In [15]:
dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False
)

epochs = 100

for epoch in range(epochs):

    lstm_model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        output = lstm_model(
            X_batch
        )

        loss = criterion(
            output,
            y_batch
        )

        if not torch.isfinite(loss):
            raise ValueError(
                "❌ LSTM loss became NaN/Inf"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1}/{epochs} "
            f"Loss: {total_loss/len(loader):.4f}"
        )

print("✅ LSTM training completed!")

Epoch 10/100 Loss: 0.0176
Epoch 20/100 Loss: 0.0070
Epoch 30/100 Loss: 0.0041
Epoch 40/100 Loss: 0.0027
Epoch 50/100 Loss: 0.0020
Epoch 60/100 Loss: 0.0015
Epoch 70/100 Loss: 0.0012
Epoch 80/100 Loss: 0.0010
Epoch 90/100 Loss: 0.0008
Epoch 100/100 Loss: 0.0007
✅ LSTM training completed!


In [16]:
lstm_model.eval()

lstm_probabilities = []

with torch.no_grad():

    for i in range(
        0,
        len(X_test_tensor),
        8
    ):

        batch = X_test_tensor[
            i:i+8
        ].to(device)

        output = lstm_model(
            batch
        )

        probability = torch.sigmoid(
            output
        )

        lstm_probabilities.append(
            probability.cpu()
        )

        del batch
        del output
        del probability

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

lstm_prob = torch.cat(
    lstm_probabilities
).numpy().flatten()

lstm_pred = (
    lstm_prob >= 0.5
).astype(int)

print("✅ LSTM prediction completed!")

✅ LSTM prediction completed!


In [17]:
lstm_accuracy = accuracy_score(
    y_test_seq,
    lstm_pred
)

lstm_precision = precision_score(
    y_test_seq,
    lstm_pred,
    zero_division=0
)

lstm_recall = recall_score(
    y_test_seq,
    lstm_pred,
    zero_division=0
)

lstm_f1 = f1_score(
    y_test_seq,
    lstm_pred,
    zero_division=0
)

if len(np.unique(y_test_seq)) == 2:
    lstm_auc = roc_auc_score(
        y_test_seq,
        lstm_prob
    )
else:
    lstm_auc = np.nan

print("LSTM Results")
print("=============")
print("Accuracy :", lstm_accuracy)
print("Precision:", lstm_precision)
print("Recall   :", lstm_recall)
print("F1 Score :", lstm_f1)
print("ROC-AUC  :", lstm_auc)

LSTM Results
Accuracy : 0.9411764705882353
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 0.5


In [18]:
class GRUModel(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=32,
        num_layers=2
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        out, _ = self.gru(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out


gru_model = GRUModel().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(),
    lr=0.001
)

print("✅ GRU created")

✅ GRU created


In [19]:
for epoch in range(100):

    gru_model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        output = gru_model(
            X_batch
        )

        loss = criterion(
            output,
            y_batch
        )

        if not torch.isfinite(loss):
            raise ValueError(
                "❌ GRU loss became NaN/Inf"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gru_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch+1}/100 "
            f"Loss: {total_loss/len(loader):.4f}"
        )

print("✅ GRU training completed!")

Epoch 10/100 Loss: 0.0120
Epoch 20/100 Loss: 0.0053
Epoch 30/100 Loss: 0.0032
Epoch 40/100 Loss: 0.0022
Epoch 50/100 Loss: 0.0016
Epoch 60/100 Loss: 0.0012
Epoch 70/100 Loss: 0.0010
Epoch 80/100 Loss: 0.0008
Epoch 90/100 Loss: 0.0007
Epoch 100/100 Loss: 0.0006
✅ GRU training completed!


In [20]:
gru_model.eval()

gru_probabilities = []

with torch.no_grad():

    for i in range(
        0,
        len(X_test_tensor),
        8
    ):

        batch = X_test_tensor[
            i:i+8
        ].to(device)

        output = gru_model(
            batch
        )

        probability = torch.sigmoid(
            output
        )

        gru_probabilities.append(
            probability.cpu()
        )

        del batch
        del output
        del probability

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

gru_prob = torch.cat(
    gru_probabilities
).numpy().flatten()

gru_pred = (
    gru_prob >= 0.5
).astype(int)

print("✅ GRU prediction completed!")

✅ GRU prediction completed!


In [21]:
gru_accuracy = accuracy_score(
    y_test_seq,
    gru_pred
)

gru_precision = precision_score(
    y_test_seq,
    gru_pred,
    zero_division=0
)

gru_recall = recall_score(
    y_test_seq,
    gru_pred,
    zero_division=0
)

gru_f1 = f1_score(
    y_test_seq,
    gru_pred,
    zero_division=0
)

if len(np.unique(y_test_seq)) == 2:

    gru_auc = roc_auc_score(
        y_test_seq,
        gru_prob
    )

else:

    gru_auc = np.nan

print("GRU Results")
print("============")
print("Accuracy :", gru_accuracy)
print("Precision:", gru_precision)
print("Recall   :", gru_recall)
print("F1 Score :", gru_f1)
print("ROC-AUC  :", gru_auc)



GRU Results
Accuracy : 0.9411764705882353
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0
ROC-AUC  : 1.0


In [22]:
results = pd.DataFrame({

    "Model": [
        "LSTM",
        "GRU",
        "XGBoost"
    ],

    "Accuracy": [
        lstm_accuracy,
        gru_accuracy,
        xgb_accuracy
    ],

    "Precision": [
        lstm_precision,
        gru_precision,
        xgb_precision
    ],

    "Recall": [
        lstm_recall,
        gru_recall,
        xgb_recall
    ],

    "F1": [
        lstm_f1,
        gru_f1,
        xgb_f1
    ],

    "ROC_AUC": [
        lstm_auc,
        gru_auc,
        xgb_auc
    ]
})

print(results)

     Model  Accuracy  Precision  Recall   F1  ROC_AUC
0     LSTM  0.941176        0.0     0.0  0.0      0.5
1      GRU  0.941176        0.0     0.0  0.0      1.0
2  XGBoost  0.941176        0.0     0.0  0.0      0.5


In [23]:
results["Recall_Rank"] = (
    results["Recall"]
    .rank(ascending=False)
)

results["F1_Rank"] = (
    results["F1"]
    .rank(ascending=False)
)

results["AUC_Rank"] = (
    results["ROC_AUC"]
    .rank(
        ascending=False,
        na_option="bottom"
    )
)

results["Accuracy_Rank"] = (
    results["Accuracy"]
    .rank(ascending=False)
)

results["Total_Rank"] = (
    results["Recall_Rank"]
    + results["F1_Rank"]
    + results["AUC_Rank"]
    + results["Accuracy_Rank"]
)

results = results.sort_values(
    "Total_Rank"
).reset_index(drop=True)

best_model_name = results.loc[
    0,
    "Model"
]

print("\n🏆 BEST MODEL:", best_model_name)

display(results)


🏆 BEST MODEL: GRU


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Recall_Rank,F1_Rank,AUC_Rank,Accuracy_Rank,Total_Rank
0,GRU,0.941176,0.0,0.0,0.0,1.0,2.0,2.0,1.0,2.0,7.0
1,LSTM,0.941176,0.0,0.0,0.0,0.5,2.0,2.0,2.5,2.0,8.5
2,XGBoost,0.941176,0.0,0.0,0.0,0.5,2.0,2.0,2.5,2.0,8.5


In [24]:
def landslide_risk_score(probability):

    return float(
        np.clip(
            probability,
            0,
            1
        )
    )

In [25]:
def landslide_risk_level(score):

    if score < 0.20:
        return "LOW"

    elif score < 0.40:
        return "MODERATE"

    elif score < 0.60:
        return "HIGH"

    elif score < 0.80:
        return "VERY HIGH"

    else:
        return "EXTREME"

In [26]:
if best_model_name == "LSTM":

    probability = float(
        lstm_prob[-1]
    )

elif best_model_name == "GRU":

    probability = float(
        gru_prob[-1]
    )

else:

    probability = float(
        xgb_prob[-1]
    )

risk_score = landslide_risk_score(
    probability
)

risk_level = landslide_risk_level(
    risk_score
)

print("================================")
print("LANDSLIDE RISK")
print("================================")
print("Best Model :", best_model_name)
print("Probability:", probability)
print("Risk Score :", risk_score)
print("Risk Level :", risk_level)

LANDSLIDE RISK
Best Model : GRU
Probability: 0.0005532411742024124
Risk Score : 0.0005532411742024124
Risk Level : LOW


In [27]:
MODEL_DIR = r"E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

torch.save({

    "model_type": "LSTM",

    "input_size": 1,

    "hidden_size": 32,

    "num_layers": 2,

    "sequence_length": 12,

    "model_state_dict":
        lstm_model.state_dict()

}, os.path.join(
    MODEL_DIR,
    "landslide_lstm_model.pt"
))

print("✅ LSTM model saved!")

✅ LSTM model saved!


In [28]:
torch.save({

    "model_type": "GRU",

    "input_size": 1,

    "hidden_size": 32,

    "num_layers": 2,

    "sequence_length": 12,

    "model_state_dict":
        gru_model.state_dict()

}, os.path.join(
    MODEL_DIR,
    "landslide_gru_model.pt"
))

print("✅ GRU model saved!")

✅ GRU model saved!


In [29]:
torch.save({

    "model_type": "XGBoost",

    "model": xgb_model,

    "features": xgb_features

}, os.path.join(
    MODEL_DIR,
    "landslide_xgboost_model.pt"
))

print("✅ XGBoost model saved!")

✅ XGBoost model saved!


In [30]:
best_model_path = os.path.join(
    MODEL_DIR,
    "landslide_best_model.pt"
)

if best_model_name == "LSTM":

    torch.save({

        "model_type": "LSTM",

        "input_size": 1,

        "hidden_size": 32,

        "num_layers": 2,

        "sequence_length": 12,

        "model_state_dict":
            lstm_model.state_dict(),

        "risk_score": risk_score,

        "risk_level": risk_level

    }, best_model_path)


elif best_model_name == "GRU":

    torch.save({

        "model_type": "GRU",

        "input_size": 1,

        "hidden_size": 32,

        "num_layers": 2,

        "sequence_length": 12,

        "model_state_dict":
            gru_model.state_dict(),

        "risk_score": risk_score,

        "risk_level": risk_level

    }, best_model_path)


elif best_model_name == "XGBoost":

    torch.save({

        "model_type": "XGBoost",

        "model": xgb_model,

        "features": xgb_features,

        "risk_score": risk_score,

        "risk_level": risk_level

    }, best_model_path)


print("================================")
print("🏆 BEST MODEL SAVED")
print("================================")
print("Model :", best_model_name)
print("Score :", risk_score)
print("Level :", risk_level)
print("Path  :", best_model_path)

🏆 BEST MODEL SAVED
Model : GRU
Score : 0.0005532411742024124
Level : LOW
Path  : E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide\landslide_best_model.pt


In [31]:
import json

metadata = {

    "dataset": "Historical Landslide",

    "best_model": best_model_name,

    "risk_score": risk_score,

    "risk_level": risk_level,

    "models": results[
        [
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC"
        ]
    ].replace(
        {np.nan: None}
    ).to_dict(
        orient="records"
    )
}

metadata_path = os.path.join(
    MODEL_DIR,
    "landslide_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("✅ Metadata saved!")
print(metadata_path)

✅ Metadata saved!
E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide\landslide_metadata.json
